# 🌿 BioCiclo IA - Treinamento YOLO Calibrado & Anti-Viés Cultural (Google Colab GPU)
**Disciplina:** Fundamentos da Inteligência Artificial - UFS  
**Contexto:** BioCiclo Urbano (Aracaju / EMSURB)  

> **Por que este notebook melhora drasticamente a precisão?**
> 1. **Rebalanceamento de Instâncias (`BIODEGRADABLE`):** No dataset *Garbage Classification 3 (GC1)*, fotos de orgânicos possuem 30 a 50 caixas por imagem, fazendo o modelo chutar `BIODEGRADABLE` para tudo. A Célula 3 corrige esse desbalanceamento antes do treino.
> 2. **Arquitetura `yolo11s.pt` + Ganho de Classe (`cls=1.5`) + `cos_lr=True`:** Força a rede neural a distinguir melhor materiais visualmente próximos (Plástico Flexível vs. Papel vs. Papelão).

**Instruções:**  
1. No menu superior do Colab, vá em **Ambiente de execução** > **Alterar tipo de ambiente de execução** > Selecione **T4 GPU**.
2. Execute as células abaixo em sequência (`Shift + Enter`).

### 1. Verificar GPU e Instalar Dependências

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

### 2. Baixar o Dataset Garbage Classification 3 v2 (GC1)

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "MCFCWzlXnoQWlcDxiV2W"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("material-identification").project("garbage-classification-3")
version = project.version(2)
dataset = version.download("yolov8")
print("Dataset baixado com sucesso em:", dataset.location)

### 3. Rebalanceamento Automático de Instâncias (Correção do Viés `BIODEGRADABLE`)
Limita o excesso de caixas repetidas em fotos superlotadas da classe `0` (`BIODEGRADABLE`), preservando 100% das imagens e de todas as classes recicláveis (`PLASTIC`, `METAL`, `GLASS`, `CARDBOARD`, `PAPER`).

In [ ]:
from pathlib import Path
from collections import Counter

train_labels = Path(dataset.location) / "train" / "labels"
before_counts = Counter()
after_counts = Counter()
MAX_BIO_BOXES = 6

for lf in train_labels.glob("*.txt"):
    lines = [ln.strip() for ln in lf.read_text().splitlines() if ln.strip()]
    parsed = []
    for ln in lines:
        parts = ln.split()
        if len(parts) >= 5:
            cid = int(parts[0])
            before_counts[cid] += 1
            parsed.append((cid, parts, ln))
    bio = [x for x in parsed if x[0] == 0]
    others = [x for x in parsed if x[0] != 0]
    if len(bio) > MAX_BIO_BOXES:
        bio.sort(key=lambda x: float(x[1][3]) * float(x[1][4]), reverse=True)
        bio = bio[:MAX_BIO_BOXES]
        final = bio + others
        lf.write_text("\n".join(x[2] for x in final) + "\n")
    else:
        final = parsed
    for cid, _, _ in final:
        after_counts[cid] += 1

print("Contagem de caixas ANTES do rebalanceamento:", dict(before_counts))
print("Contagem de caixas DEPOIS do rebalanceamento:", dict(after_counts))

### 4. Treinar o Modelo YOLO Calibrado (Fine-Tuning com Transfer Learning)
Utilizamos `yolo11s.pt` (ou `yolo11n.pt`) com `cos_lr=True`, `close_mosaic=10` e `cls=1.5` por 45 épocas.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=45,
    imgsz=640,
    batch=16,
    device=0,
    cos_lr=True,
    close_mosaic=10,
    cls=1.5,
    dropout=0.1,
    patience=12,
    project="biociclo_runs",
    name="treino_yolo_biociclo"
)

### 5. Visualizar Matriz de Confusão e Baixar `best.pt`
Após o download do arquivo `best.pt`, substitua-o na pasta `models/weights/best.pt` do seu repositório.

In [ ]:
import glob
from google.colab import files
from IPython.display import Image, display

best_pts = glob.glob('/content/**/weights/best.pt', recursive=True)
matrices = glob.glob('/content/**/confusion_matrix_normalized.png', recursive=True) or glob.glob('/content/**/confusion_matrix.png', recursive=True)
results_imgs = glob.glob('/content/**/results.png', recursive=True)

if matrices:
    display(Image(filename=matrices[0]))
if results_imgs:
    display(Image(filename=results_imgs[0]))

if best_pts:
    print(f"Iniciando download de {best_pts[0]}...")
    files.download(best_pts[0])